## Webpage Extraction and Embedding (PolyU CUS)

### 1. Extracting raw text data

In [10]:
from bs4 import BeautifulSoup
from langchain_community.document_loaders import RecursiveUrlLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import Chroma
from langchain_community.embeddings import OllamaEmbeddings

from chromadb.config import Settings
from chromadb import Client, PersistentClient

from concurrent.futures import ThreadPoolExecutor
import re

In [11]:
cus_URL = "https://www.polyu.edu.hk/cus/"
start_idx, stop_idx = 6710, -850

docs = []
unwanted_metadata = ["language"]

def bs4_regex_enhance(html: str):
    soup = BeautifulSoup(html, "lxml")                  # Strip the html syntax
    text = re.sub(r"\n\n+", "\n\n", soup.text).strip()  # Strip the excess whitespaces
    return text

cus_loader = RecursiveUrlLoader(
    url=cus_URL,
    base_url=cus_URL,
    prevent_outside=True,
    exclude_dirs=[
        cus_URL+"about-ous",
        cus_URL+"Sitemap", 
        cus_URL+"sitemap",
        cus_URL+"Search-Result", 
        cus_URL+"search-result", 
        cus_URL+"staff",
        cus_URL+"Staff",
        cus_URL+"internal",
        cus_URL+"Undergraduate-Studies-Support/Staff",
        cus_URL+"undergraduate-studies-support/staff",
    ],
    extractor=bs4_regex_enhance
)

docs_lazy = cus_loader.lazy_load()
for doc in docs_lazy:
    print(doc.metadata.get('source'))
    
    doc.page_content = doc.page_content[start_idx:stop_idx]
    for key in unwanted_metadata:
        del doc.metadata[key]
    docs.append(doc)

https://www.polyu.edu.hk/cus/
https://www.polyu.edu.hk/cus/student/4-year-undergraduate-student/academic-integrity/
https://www.polyu.edu.hk/cus/about-cus/news-and-events/ous-events/2025/3/20250312_non-local-car-briefing/
https://www.polyu.edu.hk/cus/student/4-year-undergraduate-student/admission/
https://www.polyu.edu.hk/cus/student/senior-year-intakes-and-articulation-degree-programme/academic-integrity/
https://www.polyu.edu.hk/cus/student/4-year-undergraduate-student/general-university-requirements/
https://www.polyu.edu.hk/cus/student/
https://www.polyu.edu.hk/cus/non-local-gur-study/non-local-car-subjects/subject-on-offer/
https://www.polyu.edu.hk/cus/about-cus/personal-information-collection-statement/
https://www.polyu.edu.hk/cus/ielts/
https://www.polyu.edu.hk/cus/about-cus/welcome-message/
https://www.polyu.edu.hk/cus/about-cus/news-and-events/news/
https://www.polyu.edu.hk/cus/about-cus/news-and-events/ous-events/
https://www.polyu.edu.hk/cus/about-cus/contact-us/
https://ww

In [12]:
print(f"Extracted number of webpages in CUS: {len(docs)}")
print(docs[33].page_content[:])
print(docs[33].metadata.get('source'))

'''
idx = 5
print(docs[idx].metadata.get('source'))
print(docs[idx].page_content)
#print(docs[idx].page_content[1300:-300])
'''

Extracted number of webpages in CUS: 44
ontent

													Home
												

													Non-local GUR Study
												

													Non-local Study Fund
												

													Non-local Study Fund
												

Non-local Study Fund

 
At Hong Kong Polytechnic University, we believe that an all-rounded education goes beyond academic knowledge and skills. In this rapidly changing environment, it is also important that students have a good understanding of the world and can function in environments that are different from where they come from.
For this purpose, the University has planned that by AY 2027/28, every undergraduate student will have a non-local study opportunity, half of which will achieve through Service-Learning (SL) programmes, and 10% of which will achieve through Cluster-Area Requirements (CAR) subjects. To support students in these non-local learning experiences, the University will provide financial support of up to HK$10,000 per student to cover the related

"\nidx = 5\nprint(docs[idx].metadata.get('source'))\nprint(docs[idx].page_content)\n#print(docs[idx].page_content[1300:-300])\n"

### 2. Text Splitting

In [13]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", ".", "!", "?", " ", ""],
    length_function=len,
    is_separator_regex=False,
)

chunks = text_splitter.split_documents(docs)

for i, chunk in enumerate(chunks):
    source = chunk.metadata.get("source", "N/A")
    url_end = re.search(r'/([^/]+)/?$', source).group(1)
    chunk.metadata["chunk_id"] = f"PolyU_CUS_{url_end}_chunk_{i}"

    chunk.page_content = f"--- PolyU CUS Website URL: {source} ---\n\n{chunk.page_content}"

In [14]:
print(chunks[20])

page_content='--- PolyU CUS Website URL: https://www.polyu.edu.hk/cus/ielts/ ---

Home
												

													IELTS
												

IELTS

“Effective communication” is one of the graduate attributes of PolyU undergraduate degree students. To achieve this, we strive to provide curricular and co-curricular programmes to students for enhancement of their language proficiency. To encourage students to improve their language proficiency through the preparation for an English test, and provide an internationally recognised assessment for them in pursuing further studies or entering the workforce, we are pleased to inform you that PolyU Management has decided to sponsor graduating students of undergraduate degree programmes to take the International English Language Testing System (IELTS) starting from the 2022/23 academic year.' metadata={'source': 'https://www.polyu.edu.hk/cus/ielts/', 'content_type': 'text/html; charset=utf-8', 'title': 'IELTS | College of Undergraduate Studies', 'descri

### 3. Document Embedding in Chroma

In [15]:
SINGLE = True # Change to True if you want to use single chroma database for all documents
collection_name = "academic_documents" if not SINGLE else "vaa_documents"

In [16]:
embedding_function = OllamaEmbeddings(model="bge-m3:567m") # Please OPEN Ollama first!!

client = Client(Settings())
client = PersistentClient(path="../chroma_db")
collection = client.get_collection(name=collection_name)

In [17]:
def generate_embedding(chunk):
    return embedding_function.embed_query(chunk.page_content)
with ThreadPoolExecutor() as executor:
    embeddings = list(executor.map(generate_embedding, chunks))

for i, chunk in enumerate(chunks):
    collection.add(
        documents=[chunk.page_content], 
        metadatas=[chunk.metadata], 
        embeddings=[embeddings[i]],
        ids=[str(i + 0)]
    )

print(f"Added {len(chunks)} chunks into ChromaDB to {collection_name}")

Added 112 chunks into ChromaDB to vaa_documents


### 4. Simple Testing

In [18]:
vectorStore = Chroma(
    collection_name=collection_name, 
    client=client, 
    embedding_function=embedding_function)

query = "What is the general university requirement for undergraduate student?"
results = vectorStore.similarity_search(query, k=5)

for result in results:
    print("========================================================")
    print(f"Content: {result.page_content}...")
    print(f"Source: {result.metadata.get('source')}")
    print(f"Chunk ID: {result.metadata.get('chunk_id')}\n")

Content: --- PolyU CUS Website URL: https://www.polyu.edu.hk/cus/student/4-year-undergraduate-student/general-university-requirements/ ---

ess

Start main content

													Home
												

													Student
												

													4-Year Undergraduate Student
												

													General University Requirements (GUR)
												

General University Requirements (GUR)

GUR for 4-Year Undergraduate Student 

 

Admitted in 2021/22 or before

Freshman Seminar

Language & Communication Requirements

Leadership & Intra-Personal Development

Cluster-Area Requirements

Service-Learning

Healthy Lifestyle

 

Admitted from 2022/23

Artificial Intelligence and Data Analytics Requirement

Innovation and Entrepreneurship Requirement

Language & Communication Requirements

Leadership Education and Development

Cluster-Area Requirements

Service-Learning

Healthy Lifestyle

 

For the details of curriculum framework of the General University Requirements (GUR), please click her